# Lab 22-04 · Isolated Converters

**Predict first**, then run. The handout is [README.md](README.md).

In [1]:
import collections
import math
import re
import shutil
import subprocess
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import isolated as iso


def ngspice(netlist):
    if shutil.which("ngspice") is None:
        print(f"ngspice is not installed: skipping {netlist} (the README lists the expected values)")
        return collections.defaultdict(lambda: float("nan"))
    out = subprocess.run(["ngspice", "-b", netlist], capture_output=True, text=True).stdout
    return {mm.group(1): float(mm.group(2)) for mm in re.finditer(r"^(\w+)\s*=\s*([-+0-9.eE]+)", out, re.M)}

## Part 1 · The flyback

*Predict* the duty cycle, the output and the drain voltage while the switch is off, for 150 V to 20 V with n = 6.4.

In [2]:
D = iso.flyback_duty(150, 20, 6.4)
print(f"D = {D:.3f}, reflected voltage {6.4*20:.0f} V, drain while off {iso.flyback_switch_voltage(150, 20, 6.4):.0f} V")
print(f"DCM below {iso.flyback_dcm_boundary_current(150, 20, 6.4, 100e3, 500e-6):.2f} A of output current")
s = ngspice("flyback.cir")
print({k: s[k] for k in ("vout", "vdoff")})
v_in = np.linspace(90, 375, 200)
plt.figure()
plt.plot(v_in, [iso.flyback_duty(x, 20, 6.4) for x in v_in])
plt.xlabel("bus voltage (V)"); plt.ylabel("D"); plt.title("Flyback duty cycle across the universal-input range"); plt.show()

D = 0.460, reflected voltage 128 V, drain while off 278 V
DCM below 2.38 A of output current


{'vout': 19.97522, 'vdoff': 278.0229}


## Part 2 · The forward converter and its reset

*Predict* the output, the drain voltage during reset and when the reset ends, for 48 V to 5 V with 3:1:3 windings.

In [3]:
print(f"D = {iso.forward_duty(48, 5, 3):.3f}, D_max = {iso.forward_max_duty(1.0):.2f}, "
      f"drain during reset {iso.forward_switch_voltage(48, 1.0):.0f} V")
s = ngspice("forward.cir")
print({k: s[k] for k in ("vout", "vdreset", "imend")})
# An idealized period from the formulas: magnetizing current ramps up for D*Ts, then the tertiary ramps it down equally fast.
Ts, D, Lm = 4e-6, iso.forward_duty(48, 5, 3), 300e-6   # the netlist: 250 kHz, 300 µH primary
t = np.linspace(0, Ts, 1000)
im = np.where(t < D * Ts, 48 * t / Lm, np.maximum(48 * D * Ts / Lm - 48 * (t - D * Ts) / Lm, 0))
vd = np.where(t < D * Ts, 0, np.where(im > 0, 96, 48))
fig, ax = plt.subplots(2, 1, sharex=True)
ax[0].plot(t * 1e6, vd); ax[0].set_ylabel("drain (V)")
ax[1].plot(t * 1e6, im * 1e3); ax[1].set_ylabel("i_m (mA)"); ax[1].set_xlabel("t (µs)")
plt.show()

D = 0.312, D_max = 0.50, drain during reset 96 V


{'vout': 4.979722, 'vdreset': 96.0132, 'imend': 4.802466e-08}


## Part 3 · The cost of leakage

*Predict* the clamp power for 1%, 2% and 5% leakage in Example 22-04.2's flyback (L_m = 223 µH, I_pk = 2.52 A, 100 kHz) with a 200 V clamp.

In [4]:
d, ipk, lm = iso.flyback_boundary_design(65 / 0.92, 100, 20, 6.4, 100e3)
print(f"boundary design: D = {d:.3f}, I_pk = {ipk:.2f} A, L_m = {lm*1e6:.0f} µH")
for frac in (0.01, 0.02, 0.05):
    p = iso.clamp_energy(frac * lm, ipk, 200, 128) * 100e3
    print(f"{frac:.0%} leakage: clamp power {p:.2f} W ({p/(65/0.92)*100:.1f} % of the input)")
vc = np.linspace(140, 400, 200)
plt.figure()
for frac in (0.01, 0.02, 0.05):
    plt.plot(vc, [iso.clamp_energy(frac * lm, ipk, x, 128) * 100e3 for x in vc], label=f"{frac:.0%} leakage")
plt.xlabel("clamp voltage above the bus (V)"); plt.ylabel("clamp power (W)"); plt.legend(); plt.show()

boundary design: D = 0.561, I_pk = 2.52 A, L_m = 223 µH
1% leakage: clamp power 1.96 W (2.8 % of the input)
2% leakage: clamp power 3.93 W (5.6 % of the input)
5% leakage: clamp power 9.81 W (13.9 % of the input)


## Part 4 · Insulation arithmetic

*Predict* the Y-capacitance limit for 0.5 mA at 264 V (50 and 60 Hz) and the reinforced clearance at sea level and 5000 m.

In [5]:
for f in (50, 60):
    print(f"{f} Hz: Y-capacitance limit {iso.y_cap_limit(0.5e-3, 264, f)*1e9:.2f} nF")
print(f"reinforced clearance: 3.8 mm at sea level, {3.8*1.48:.2f} mm at 5000 m (altitude factor 1.48)")
for group, basic in (("I", 2.0), ("II", 2.8), ("III", 4.0)):
    print(f"material group {group}: basic creepage {basic} mm, reinforced {2*basic:.1f} mm (400 V, PD2)")

50 Hz: Y-capacitance limit 6.03 nF
60 Hz: Y-capacitance limit 5.02 nF
reinforced clearance: 3.8 mm at sea level, 5.62 mm at 5000 m (altitude factor 1.48)
material group I: basic creepage 2.0 mm, reinforced 4.0 mm (400 V, PD2)
material group II: basic creepage 2.8 mm, reinforced 5.6 mm (400 V, PD2)
material group III: basic creepage 4.0 mm, reinforced 8.0 mm (400 V, PD2)


## Discussion

Compare your predictions with the results, then answer the discussion questions in the README.